# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library in Python. All data entities are referenced by their `@id` fields as defined in the Croissant schema.

### Dataset Source
This dataset is described by a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available in the environment
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect its basic information using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Show some high-level metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}\n\nPublication Date: {metadata.datePublished}")
print(f"\nLicense: {metadata.license}\nCite as: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Enumerate available Record Sets (`cr:RecordSet`), their fields (`cr:Field`), and columns. All are identified by their `@id` fields.

In [ ]:
# Discover available record sets
recordsets = dataset.metadata.recordSets if hasattr(dataset.metadata, 'recordSets') else dataset.metadata.recordSet
# NOTE: Some datasets use `recordSets`, older versions use `recordSet` for listings.
if isinstance(recordsets, dict):
    # Single record set
    recordsets = [recordsets]
elif recordsets is None:
    # Try to get all via dataset.record_sets() as fallback
    recordsets = list(dataset.record_sets())

print(f"Record sets found (count={len(recordsets)}):")
record_set_ids = []
for i, rs in enumerate(recordsets):
    rsid = getattr(rs, '@id', None) or getattr(rs, 'id', None) or rs
    record_set_ids.append(rsid)
    print(f"  {i}: {rsid}")
    # Show fields (by id/name)
    fields = getattr(rs, 'fields', [])
    if not fields:
        fields = getattr(rs, 'field', [])
    if fields and hasattr(fields[0], '@id'):
        field_ids = [f"    - {getattr(f, '@id', str(f))} ({getattr(f, 'name', 'unnamed')})" for f in fields]
        print("    Fields:")
        print("\n".join(field_ids))
    else:
        print("    <Fields for this RecordSet not found or not detailed>")
if not record_set_ids:
    print("No record sets found in metadata; will attempt to find them via dataset.record_sets().")
    # mlcroissant provides a dataset.record_sets() iterator
    record_set_ids = [r for r in dataset.record_sets()]

To print a sample of records from a chosen record set, specify the `@id` value of the record set below.

_(For this dataset, review the printed `@id`s above and pick a record set to explore.)_

In [ ]:
# Print a sample of records from a chosen record set by its @id
# For this dataset there may only be one main record set: e.g., 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034#PROTEIN'
example_record_set_id = None
if record_set_ids:
    example_record_set_id = record_set_ids[0]  # Take the first one for demonstration
    print(f"\n--- Sample records from Record Set @id: {example_record_set_id} ---")
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        if i > 2:
            break
        pprint.pprint(x)
else:
    print("No record set found to enumerate records.")

## 3. Data Extraction
Load one or more record sets into Pandas DataFrames for further processing. Use the `@id`s of record sets and fields as shown above.

In [ ]:
# Choose record sets to load (by @id)
# If only one record set is present, demonstrate with it.
if not record_set_ids:
    raise ValueError('No record set @id found to extract.')
record_sets_to_load = record_set_ids  # Or subset, e.g. [record_set_ids[0]]
dataframes = {}
for rec_id in record_sets_to_load:
    print(f"Loading data from Record Set @id: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Columns in {rec_id}:")
    print(df.columns.tolist())
    print(df.head(2))  # Preview first records
    print()
# For the rest of notebook, select main record set
main_record_set_id = record_sets_to_load[0]

## 4. Exploratory Data Analysis (EDA)
Select numerical and categorical fields for EDA, using their `@id` as column keys. For demonstration, we'll filter, normalize, and group on a numeric field.

In [ ]:
# Pick a numeric field by inspecting column names (see above).
# For demonstration, assume a column 'MW [kDa]' for molecular weight exists (adjust if needed).
df = dataframes[main_record_set_id]
# Auto-detect a numeric field
potential_numeric_fields = [c for c in df.columns if ('MW' in c or 'mass' in c.lower() or 'weight' in c.lower() or 'abundance' in c.lower())]
if potential_numeric_fields:
    numeric_field = potential_numeric_fields[0]
else:
    # Fall back to first numeric column
    numeric_field = df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: {numeric_field}")

# Thresholding: keep records with MW > threshold (arbitrary example: 10 kDa)
try:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered entries with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the chosen numeric column
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized values for '{numeric_field}':")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    # Try possible choices: 'sample', 'experiment', 'description', etc.
    possible_group_fields = [c for c in df.columns if any(substr in c.lower() for substr in ["sample", "desc", "group", "type"])]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical field found for grouping.")
except Exception as e:
    print(f"EDA error: {e}")

## 5. Visualization
Plot the distribution of a selected numeric field and, if available, group means for a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # Plot grouped if applicable
    if 'grouped_df' in locals() and group_field in grouped_df.columns and numeric_field in grouped_df.columns:
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- The Mass Spectrometry dataset was loaded and explored using [mlcroissant](https://github.com/mlcommons/croissant).
- The main record set and its fields were identified by their Croissant `@id`.
- We filtered, normalized, and visualized a key numeric field (e.g., molecular weight).
- For more advanced analyses (e.g., protein abundance comparisons or peptide coverage insights), use field `@id` as shown and tailor data processing accordingly.

**Next steps:** Consult the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for additional field descriptions and standardized identifiers to conduct domain-specific analyses.